# F-MI-0 Angle-Separation Multifrequency Baseline

This notebook measures how reconstruction error changes as two active sources move closer together in a noiseless, multiple-frequency, overdetermined frequency-domain system.

This is a conditioning/reconstruction baseline, not a support-detection benchmark.

## Setup

- Tag: `F-MI-0-RES`
- Sources: `S = 2`, both active
- Microphones: `M = 10`
- Frequencies: all retained positive FFT bins at or above `100 Hz`
- Noise: none
- Source magnitudes: fixed across frequencies and separations
- Source phases: varied across 15 phase seeds
- Group structure: disabled with `grouping="none"` for Group LASSO and Sparse Prior

In [ ]:
from pathlib import Path
import os
import sys


def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "src" / "cs_priors").exists():
            return path
    raise RuntimeError("Could not find repository root containing src/cs_priors")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/cs_priors_matplotlib")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cs_priors.simulation.mixing_model import quick_frequency_sim
from cs_priors.solvers.freq_lasso import frequency_lasso_solve
from cs_priors.solvers.freq_group_lasso import frequency_group_lasso_solve
from cs_priors.solvers.freq_sparse_prior import sparse_prior_solve
from cs_priors.solvers.representations import mixing_tensor_to_frequency_major_matrix
from cs_priors.plotting.plot_complex import plot_matrices
from cs_priors.plotting.plot_geometry import plot_sim_geometry

In [ ]:
TAG = "F-MI-0-RES"
FIGURE_DIR = REPO_ROOT / "results" / "benchmarks" / TAG 
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


def save_figure(fig, stem: str):
    fig.savefig(FIGURE_DIR / f"{stem}.pdf", bbox_inches="tight")

NUM_MICS = 10
NUM_SOURCES = 2
NUM_ACTIVE = 2
MIN_FREQ_HZ = 100.0
SAMPLING_RATE = 2000.0
DURATION = 0.05
COMPONENT_AMPLITUDE = 1.0

MIC_RADIUS = 0.3
MIC_ANGLE_START = 0
MIC_ANGLE_SPAN = 2 * np.pi 
SOURCE_DISTANCE = 1.5

SEPARATIONS_DEG = np.array([
    1e-10, 3e-10,
    1e-9, 3e-9,
    1e-8, 3e-8,
    1e-7, 3e-7,
    1e-6, 3e-6,
    1e-5, 3e-5,
    1e-4, 3e-4,
    1e-3,
])

PHASE_SEEDS = np.arange(15)

LASSO_ALPHA = 1e-8
GROUP_LASSO_ALPHA = 1e-8
LASSO_MAX_ITER = 1_000
GROUP_LASSO_MAX_ITER = 500

METHOD_ORDER = ["r-LASSO", "Group LASSO", "Sparse Prior", "X_pinv"]
METHOD_COLORS = {
    "X_pinv": "tab:blue",
    "r-LASSO": "tab:orange",
    "Group LASSO": "tab:green",
    "Sparse Prior": "tab:red",
}
METHOD_LINESTYLES = {
    "X_pinv": ":",
    "r-LASSO": "-",
    "Group LASSO": "-",
    "Sparse Prior": "-",
}

PLOT_FLOOR = 1e-16

In [ ]:
def predict(A: np.ndarray, X: np.ndarray) -> np.ndarray:
    return np.einsum("msf,sf->mf", A, X)


def relative_complex_error(X_hat: np.ndarray, X_true: np.ndarray) -> float:
    return np.linalg.norm(X_hat - X_true) / np.linalg.norm(X_true)


def relative_residual_norm(Y: np.ndarray, A: np.ndarray, X_hat: np.ndarray) -> float:
    return np.linalg.norm(Y - predict(A, X_hat)) / np.linalg.norm(Y)


def condition_number(A: np.ndarray) -> float:
    return np.linalg.cond(mixing_tensor_to_frequency_major_matrix(A))


def make_simulation(separation_deg: float, phase_seed: int):
    separation_rad = np.deg2rad(separation_deg)
    return quick_frequency_sim(
        num_mics=NUM_MICS,
        num_sources=NUM_SOURCES,
        num_active=NUM_ACTIVE,
        seed=int(phase_seed),
        sampling_rate=SAMPLING_RATE,
        duration=DURATION,
        source_distance=SOURCE_DISTANCE,
        mic_radius=MIC_RADIUS,
        array_type="arc",
        mic_angle_start=MIC_ANGLE_START,
        mic_angle_span=MIC_ANGLE_SPAN,
        source_angle_start=-separation_rad / 2,
        source_angle_span=separation_rad,
        component_amplitude=COMPONENT_AMPLITUDE,
        magnitude_jitter=0.0,
        min_freq_hz=MIN_FREQ_HZ,
        single_frequency_hz=None,
        sensor_snr_db=None,
        model_snr_db=None,
        inverse_method="mp",
    )


def solve_methods(sim) -> dict[str, np.ndarray]:
    X_pinv = sim.X_pinv
    return {
        "X_pinv": X_pinv,
        "r-LASSO": frequency_lasso_solve(
            sim.Y,
            sim.A,
            alpha=LASSO_ALPHA,
            max_iter=LASSO_MAX_ITER,
            X_start=X_pinv,
        ),
        "Group LASSO": frequency_group_lasso_solve(
            sim.Y,
            sim.A,
            alpha=GROUP_LASSO_ALPHA,
            grouping="none",
            max_iter=GROUP_LASSO_MAX_ITER,
            X_start=X_pinv,
        ),
        "Sparse Prior": sparse_prior_solve(
            X_pinv,
            sim.A,
            grouping="none",
        ),
    }


def run_case(separation_deg: float, phase_seed: int) -> list[dict]:
    sim = make_simulation(separation_deg, phase_seed)
    cond = condition_number(sim.A)

    rows = []
    for method, X_hat in solve_methods(sim).items():
        rows.append(
            {
                "tag": TAG,
                "separation_deg": separation_deg,
                "phase_seed": phase_seed,
                "num_frequencies": sim.A.shape[2],
                "method": method,
                "relative_complex_error": relative_complex_error(X_hat, sim.X),
                "relative_residual_norm": relative_residual_norm(sim.Y, sim.A, X_hat),
                "condition_number": cond,
            }
        )
    return rows

In [ ]:
sim_check = make_simulation(SEPARATIONS_DEG[-1], PHASE_SEEDS[0])
assert sim_check.A.shape[0] == NUM_MICS
assert sim_check.A.shape[1] == NUM_SOURCES
assert sim_check.A.shape[2] > 1
assert np.all(sim_check.freqs >= MIN_FREQ_HZ)
assert np.allclose(sim_check.eta, 0.0)
assert np.allclose(sim_check.delta, 0.0)

sim_check.A.shape, sim_check.Y.shape, sim_check.X.shape, sim_check.freqs[:5], sim_check.freqs[-5:]

In [ ]:
sim = make_simulation(SEPARATIONS_DEG[0], PHASE_SEEDS[0])
solutions = solve_methods(sim)

plot_matrices(
    [sim.X, *solutions.values()],
    ["X_true", *solutions.keys()],
    show_values=False,
    dpi=90,
)

geometry_sim = make_simulation(SEPARATIONS_DEG[-1], PHASE_SEEDS[0])
fig, ax = plot_sim_geometry(geometry_sim, show=False)
save_figure(fig, "simulation_geometry")
plt.show()

In [ ]:
rows = []
for separation_deg in SEPARATIONS_DEG:
    for phase_seed in PHASE_SEEDS:
        rows.extend(run_case(float(separation_deg), int(phase_seed)))

results_df = pd.DataFrame(rows)
results_df["method"] = pd.Categorical(
    results_df["method"], categories=METHOD_ORDER, ordered=True
)
results_df = results_df.sort_values(["separation_deg", "phase_seed", "method"])
results_df.head()

In [ ]:
summary_df = (
    results_df.groupby(["method", "separation_deg"], observed=True)
    .agg(
        median_relative_complex_error=("relative_complex_error", "median"),
        q25_relative_complex_error=("relative_complex_error", lambda x: x.quantile(0.25)),
        q75_relative_complex_error=("relative_complex_error", lambda x: x.quantile(0.75)),
        median_relative_residual_norm=("relative_residual_norm", "median"),
        q25_relative_residual_norm=("relative_residual_norm", lambda x: x.quantile(0.25)),
        q75_relative_residual_norm=("relative_residual_norm", lambda x: x.quantile(0.75)),
        condition_number=("condition_number", "first"),
        num_frequencies=("num_frequencies", "first"),
    )
    .reset_index()
)
summary_df

In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 4.4))

for method in METHOD_ORDER:
    method_df = summary_df[summary_df["method"] == method].sort_values("separation_deg")
    x = method_df["separation_deg"].to_numpy(dtype=float)
    median = np.maximum(
        method_df["median_relative_complex_error"].to_numpy(dtype=float), PLOT_FLOOR
    )
    q25 = np.maximum(
        method_df["q25_relative_complex_error"].to_numpy(dtype=float), PLOT_FLOOR
    )
    q75 = np.maximum(
        method_df["q75_relative_complex_error"].to_numpy(dtype=float), PLOT_FLOOR
    )

    ax.plot(
        x,
        median,
        marker="o",
        label=method,
        color=METHOD_COLORS[method],
        linestyle=METHOD_LINESTYLES[method],
    )
    ax.fill_between(x, q25, q75, color=METHOD_COLORS[method], alpha=0.18, linewidth=0)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Angular separation (degrees)")
ax.set_ylabel("Relative complex error")
ax.set_title("Multifrequency reconstruction error vs. source angular separation")
ax.grid(True, which="both", linewidth=0.5, alpha=0.35)
ax.legend(frameon=False)

save_figure(fig, "relative_complex_error_vs_angle_separation")
plt.show()

In [ ]:
condition_df = (
    results_df[["separation_deg", "condition_number"]]
    .drop_duplicates()
    .sort_values("separation_deg")
)

fig, ax = plt.subplots(figsize=(6.2, 4.0))
ax.plot(
    condition_df["separation_deg"],
    condition_df["condition_number"],
    marker="o",
    color="black",
)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Angular separation (degrees)")
ax.set_ylabel("Condition number of A_big")
ax.set_title("Multifrequency block-matrix conditioning")
ax.grid(True, which="both", linewidth=0.5, alpha=0.35)

save_figure(fig, "condition_number_vs_angle_separation")
plt.show()

In [ ]:
example_separations = [SEPARATIONS_DEG[-1], SEPARATIONS_DEG[0]]
example_seed = 0
example_matrices = []
example_titles = []

for separation_deg in example_separations:
    sim = make_simulation(float(separation_deg), example_seed)
    solutions = {"X_true": sim.X, **solve_methods(sim)}
    for name, X in solutions.items():
        example_matrices.append(X)
        example_titles.append(f"{name}\n{separation_deg:g} deg")

plot_matrices(
    example_matrices,
    example_titles,
    show_values=False,
    dpi=90,
)
save_figure(plt.gcf(), "example_matrices_easy_and_hard_separations")
plt.show()

## Interpretation note

This notebook answers: as two active sources become angularly close, when do the methods stop reconstructing the true coefficients accurately in a noiseless multifrequency system?

It does not answer support detection or localization among many candidate source rows. Since `M=10`, `S=2`, and the retained multifrequency system is overdetermined, `Sparse Prior` is expected to return the pseudoinverse initializer unless numerical rank deficiency appears. With `grouping="none"`, the Group LASSO and Sparse Prior methods intentionally do not use cross-frequency structure.